In [ ]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import matplotlib.pyplot as plt
import seaborn as sns

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

## EDA

In [ ]:
df=pd.read_csv('/kaggle/input/taxi-price-prediction/taxi_trip_pricing.csv')
df.head(10)

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
df.describe()

In [ ]:
df['Time_of_Day'].unique()

In [ ]:
df['Day_of_Week'].unique(),


In [ ]:
df['Weather'].unique()

In [ ]:
sns.heatmap(df.corr(numeric_only=True),annot=True,cmap='coolwarm')

In [ ]:
plt.style.use('dark_background')
plt.figure(figsize=(14,10))
plt.subplot(2,2,1)
sns.countplot(df['Weather'])
plt.title('current weather')
plt.subplot(2,2,2)
sns.countplot(df['Time_of_Day'])
plt.title('time of day')
plt.subplot(2,2,3)
sns.countplot(df['Traffic_Conditions'])
plt.title('traffic dense')
plt.subplot(2,2,4)
sns.countplot(df['Day_of_Week'])
plt.title('day of week')

In [ ]:
plt.scatter(x=df['Trip_Distance_km'],y=df['Trip_Price'])
plt.title('Trip Distance and Price Relationship')

In [ ]:
sns.histplot(df['Trip_Price'],kde=True)

In [ ]:
sns.boxplot(x=df['Trip_Price'])

In [ ]:
num=df[['Trip_Distance_km','Passenger_Count','Base_Fare','Per_Km_Rate','Per_Minute_Rate','Trip_Duration_Minutes','Trip_Price']]

In [ ]:
plt.figure(figsize=(12,10))
sns.pairplot(df.select_dtypes(include='number'))

### Filling nan columns

In [ ]:
df=df.fillna(df.median(numeric_only=True))

## Preprocessing

In [ ]:
num_cols=[
    "Trip_Distance_km",
    "Trip_Duration_Minutes",
    "Base_Fare",
    "Per_Km_Rate",
    "Per_Minute_Rate"
    
]

In [ ]:
cat_cols=["Time_of_Day","Day_of_Week","Traffic_Conditions","Weather"]

In [ ]:
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import ElasticNetCV
from sklearn.linear_model import LassoCV
from sklearn.linear_model import RidgeCV

In [ ]:
preprocessor=ColumnTransformer(transformers=[
    ('num',StandardScaler(),num_cols),
    ('cat',OneHotEncoder(handle_unknown='ignore'),cat_cols)
])

In [ ]:
pipeline=Pipeline(steps=[
    ('preprocess',preprocessor),
    ('lr',LinearRegression())
])

In [ ]:
pipeline

In [ ]:
X=df.drop('Trip_Price',axis=1)
y=df['Trip_Price']

In [ ]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.2,random_state=15)

In [ ]:
pipeline.fit(X_train,y_train)

In [ ]:
y_pred=pipeline.predict(X_test)

In [ ]:
from sklearn.metrics import r2_score,mean_absolute_error,mean_squared_error
score=r2_score(y_test,y_pred)
mae=mean_absolute_error(y_test,y_pred)
mse=mean_squared_error(y_test,y_pred)


In [ ]:
print(score,mae,mse)